# Employee Attrition Prediction

## 1 · Environment Setup & Data Upload

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from google.colab import files
import io

warnings.filterwarnings("ignore")

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

if "sales" in df.columns:
    df = df.rename(columns={"sales": "Department"})

df.columns = [c.strip() for c in df.columns]
print("Data loaded. Shape:", df.shape)

## 2 · Data Preview & Summary

In [ ]:
df.info()

print(df.describe())

print(df.sample(10, random_state=42))

print("Attrition Rate:", df["left"].mean())

## 3 · Feature Distributions

In [ ]:
numeric_cols = ["satisfaction_level", "last_evaluation", "number_project", "average_montly_hours", "time_spend_company"]
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], kde=True, color="blue")
    plt.title(f"Distribution of {col}")
    plt.show()

cat_cols = ["Department", "salary", "Work_accident", "promotion_last_5years"]
for col in cat_cols:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x=col, palette="Set2")
    plt.title(f"Frequency of {col}")
    plt.xticks(rotation=45)
    plt.show()

## 4 · Features vs. Left

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df, x="left", y=col, palette="Set1")
    plt.title(f"{col} vs Left")
    plt.show()

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=col, hue="left", palette="Set1")
    plt.title(f"{col} vs Left")
    plt.xticks(rotation=45)
    plt.show()

## 5 · Correlation Heatmap

In [ ]:
df_corr = df.copy()
df_corr["salary"] = df_corr["salary"].map({"low": 0, "medium": 1, "high": 2})

df_numeric = df_corr.select_dtypes(include=[np.number])

plt.figure(figsize=(8, 6))
sns.heatmap(df_numeric.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 6 · Preprocessing & Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_model = df.copy()

df_model["salary"] = df_model["salary"].map({"low": 0, "medium": 1, "high": 2})
df_model = pd.get_dummies(df_model, columns=["Department"], prefix="dept", dtype=int)

y = df_model["left"].values
X = df_model.drop(columns=["left"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scale_cols = ["satisfaction_level", "last_evaluation", "number_project", "average_montly_hours", "time_spend_company"]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

## 7 · K-Means Clustering & PCA

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

inertias = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(range(1, 9), inertias, "o-", color="red")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(pca_data[:, 0], pca_data[:, 1], c=clusters, cmap="viridis", s=15, alpha=0.7)
plt.title("PCA Cluster Plot")
plt.colorbar(label="Cluster")
plt.show()

df_profile = df.copy()
df_profile["cluster"] = clusters

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_profile, x="satisfaction_level", y="last_evaluation", hue="cluster", style="left", palette="viridis")
plt.title("Cluster Profiles")
plt.show()

print(df_profile.groupby("cluster")[["satisfaction_level", "last_evaluation", "average_montly_hours", "number_project", "time_spend_company", "left"]].mean())

## 8 · MLP Back-Propagation Model

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=400,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)

plt.figure(figsize=(6, 4))
plt.plot(mlp.loss_curve_)
plt.title("MLP Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

mlp_pred = mlp.predict(X_test_scaled)

## 9 · Gradient Boosting Classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbm = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gbm.fit(X_train_scaled, y_train)

gbm_pred = gbm.predict(X_test_scaled)
print("GBM Trained.")

## 10 · Model Evaluation & Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

models = ["MLP (Back-Prop)", "Gradient Boosting"]
predictions = [mlp_pred, gbm_pred]

for model, pred in zip(models, predictions):
    print(f"=== {model} ===")
    print("Accuracy :", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall   :", recall_score(y_test, pred))
    print("F1-Score :", f1_score(y_test, pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, pred))
    print("\nClassification Report:")
    print(classification_report(y_test, pred))
    print("-" * 40)

## 11 · ROC Curves & Feature Importance

In [ ]:
from sklearn.metrics import roc_curve, auc

mlp_probs = mlp.predict_proba(X_test_scaled)[:, 1]
gbm_probs = gbm.predict_proba(X_test_scaled)[:, 1]

fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_probs)
auc_mlp = auc(fpr_mlp, tpr_mlp)

fpr_gbm, tpr_gbm, _ = roc_curve(y_test, gbm_probs)
auc_gbm = auc(fpr_gbm, tpr_gbm)

plt.figure(figsize=(7, 5))
plt.plot(fpr_mlp, tpr_mlp, label=f"MLP (AUC = {auc_mlp:.3f})")
plt.plot(fpr_gbm, tpr_gbm, label=f"GBM (AUC = {auc_gbm:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

importances = pd.Series(gbm.feature_importances_, index=X.columns)
top_10 = importances.sort_values().tail(10)

plt.figure(figsize=(8, 5))
top_10.plot(kind="barh", color="teal")
plt.title("Feature Importances")
plt.show()